## 🎯 Learning Objectives
* Understand the critical role of data preparation, ingestion, and indexing in building effective Agentic RAG systems for e-commerce.
* Learn various strategies and tools for acquiring, cleaning, and structuring product catalog data.
* Implement a practical pipeline for generating vector embeddings from product descriptions and indexing them in a vector database.
* Evaluate the trade-offs and performance considerations associated with different data preparation and indexing approaches.


## Data Preparation: Product Catalogue Ingestion and Indexing

Welcome to PRJ02-L02, where we dive into the foundational step for any robust Agentic RAG system: preparing your data. For our e-commerce project, this means meticulously handling the product catalog. Think of your e-commerce store as a vast library. Before anyone can find a book, the library needs to acquire the books (ingestion), categorize them, and create an efficient catalog system (indexing). Without this, even the smartest librarian (our AutoGen agents) would struggle to find what you're looking for.

### Why is Data Preparation Crucial for Agentic RAG?

In the world of Retrieval Augmented Generation (RAG), the quality of your retrieved information directly impacts the quality of the generated response. This is often summarized as "Garbage In, Garbage Out" (GIGO). For agentic systems, where autonomous agents rely on accurate and relevant information to make decisions and formulate responses, well-prepared data is paramount. It ensures:

1.  **Accuracy**: Agents retrieve correct product details, specifications, and pricing.
2.  **Relevance**: Semantic search capabilities allow agents to find products even with nuanced or indirect queries.
3.  **Efficiency**: Optimized indexing enables rapid retrieval, crucial for real-time agent interactions.
4.  **Scalability**: A structured approach allows the system to handle millions of products without performance degradation.

### Product Catalogue Ingestion: Bringing Data In

Ingestion is the process of acquiring raw product data from various sources and bringing it into a usable format. E-commerce product data can originate from:

*   **Databases**: SQL (PostgreSQL, MySQL), NoSQL (MongoDB, Cassandra) – often the primary source for structured product information.
*   **APIs**: Vendor APIs, supplier feeds, third-party data providers.
*   **Files**: CSV, JSON, XML files containing product listings.
*   **Web Scraping**: For competitor analysis or augmenting existing data (though often legally and ethically complex).

**Challenges during Ingestion:**

*   **Data Cleaning**: Removing inconsistencies, duplicates, missing values, and formatting errors.
*   **Schema Mapping**: Harmonizing data from disparate sources into a unified schema.
*   **Text Extraction**: Extracting meaningful text from rich HTML descriptions or PDFs.
*   **Data Enrichment**: Adding supplementary information (e.g., product reviews, FAQs) to enhance context.

Modern tools like `pandas` for data manipulation, `Pydantic` for data validation, and dedicated ETL (Extract, Transform, Load) pipelines are essential for robust ingestion.

### Product Catalogue Indexing: Making Data Searchable

Once ingested and cleaned, the data needs to be indexed. Indexing transforms raw data into a format optimized for fast and relevant retrieval. For RAG, this primarily involves **vector indexing** for semantic search.

**Key Indexing Concepts:**

1.  **Text Chunking**: Breaking down long product descriptions or specifications into smaller, manageable chunks. While less critical for short product descriptions, it's vital for longer documents like user manuals or detailed reviews.
2.  **Embedding Generation**: Converting text chunks (or entire product descriptions) into high-dimensional numerical vectors (embeddings). These vectors capture the semantic meaning of the text, allowing us to find semantically similar items by calculating vector distance.
    *   **Embedding Models**: State-of-the-art models like `BAAI/bge-large-en-v1.5` (from Hugging Face's `sentence-transformers` library) or proprietary models from OpenAI, Cohere, and Google are commonly used in 2026.
3.  **Vector Database Storage**: Storing these embeddings along with their associated metadata (product ID, name, price, category) in a specialized vector database. These databases are optimized for fast similarity search across millions or billions of vectors.
    *   **Popular Vector Databases (2026)**: Pinecone, Weaviate, Qdrant, ChromaDB, Milvus, and cloud-native solutions like Azure AI Search (with vector capabilities) or Google Cloud's Vertex AI Vector Search.

**Step-by-Step Indexing Process:**

1.  **Load Data**: Read your cleaned product catalog data.
2.  **Prepare Text**: Concatenate relevant text fields (e.g., `product_name` + `description` + `features`) into a single string for embedding.
3.  **Generate Embeddings**: Pass the prepared text through an embedding model to get vector representations.
4.  **Store in Vector DB**: Insert the generated embeddings, along with original product metadata, into your chosen vector database.
5.  **Query**: When a user asks a question, embed the query and perform a similarity search in the vector database to retrieve the most relevant product information.

This structured approach ensures that our AutoGen agents have a rich, semantically searchable knowledge base to draw upon, enabling them to provide accurate and intelligent responses to customer queries.


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
import uuid

# --- 1. Simulate Product Catalog Data --- 
# In a real scenario, this would come from a database, API, or file.
print("1. Generating sample product catalog data...")
product_data = [
    {
        "product_id": "P001",
        "name": "Smartwatch Xtreme 2026",
        "category": "Wearables",
        "description": "The latest smartwatch with advanced health tracking, 5G connectivity, and a 3-day battery life. Features a vibrant AMOLED display and waterproof design. Perfect for fitness enthusiasts and tech-savvy users.",
        "price": 399.99
    },
    {
        "product_id": "P002",
        "name": "Eco-Friendly Reusable Coffee Cup",
        "category": "Kitchenware",
        "description": "Sustainable coffee cup made from recycled materials. Double-walled insulation keeps drinks hot for hours. Leak-proof lid and ergonomic design. Available in multiple colors.",
        "price": 19.99
    },
    {
        "product_id": "P003",
        "name": "AI-Powered Home Security Camera",
        "category": "Smart Home",
        "description": "Next-gen security camera with AI-driven motion detection, facial recognition, and cloud storage. 4K video resolution and two-way audio. Integrates seamlessly with smart home ecosystems.",
        "price": 149.99
    },
    {
        "product_id": "P004",
        "name": "Ergonomic Office Chair Pro",
        "category": "Office Furniture",
        "description": "Designed for ultimate comfort and posture support during long working hours. Adjustable lumbar support, headrest, and armrests. High-quality breathable mesh fabric. A must-have for any modern workspace.",
        "price": 299.00
    },
    {
        "product_id": "P005",
        "name": "Wireless Noise-Cancelling Headphones",
        "category": "Audio",
        "description": "Immersive audio experience with industry-leading noise cancellation. 30-hour battery life, comfortable over-ear design, and crystal-clear call quality. Perfect for travel and focused work.",
        "price": 249.50
    }
]

df = pd.DataFrame(product_data)
print("Sample Data Head:")
print(df.head())
print("\n" + "-"*50 + "\n")

# --- 2. Data Preprocessing and Text Preparation --- 
# Combine relevant text fields into a single string for embedding.
# This step is crucial for creating a comprehensive semantic representation.
print("2. Preprocessing data and preparing text for embedding...")
def prepare_product_text(row):
    return f"Product Name: {row['name']}. Category: {row['category']}. Description: {row['description']}"

df['full_text_description'] = df.apply(prepare_product_text, axis=1)
print("Example of prepared text for P001:")
print(df.loc[df['product_id'] == 'P001', 'full_text_description'].iloc[0])
print("\n" + "-"*50 + "\n")

# --- 3. Initialize Embedding Model --- 
# Using a state-of-the-art sentence transformer model from Hugging Face.
# 'BAAI/bge-small-en-v1.5' is a good balance of performance and speed for many applications.
print("3. Initializing embedding model (BAAI/bge-small-en-v1.5). This may take a moment...")
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
print("Embedding model loaded successfully.")
print("\n" + "-"*50 + "\n")

# --- 4. Initialize ChromaDB Client and Collection --- 
# ChromaDB is an open-source vector database that can run in-memory or persistently.
# For this example, we'll use a persistent client to demonstrate basic usage.
print("4. Initializing ChromaDB client and collection...")
client = chromadb.PersistentClient(path="./chroma_db") # Stores data in a local directory

# Ensure a clean slate for demonstration
try:
    client.delete_collection(name="ecommerce_products")
    print("Existing 'ecommerce_products' collection deleted.")
except:
    pass # Collection might not exist yet

collection = client.get_or_create_collection(name="ecommerce_products")
print(f"ChromaDB collection '{collection.name}' ready.")
print("\n" + "-"*50 + "\n")

# --- 5. Generate Embeddings and Index Data in ChromaDB --- 
print("5. Generating embeddings and indexing products in ChromaDB...")

# Prepare data for ChromaDB insertion
documents = df['full_text_description'].tolist()
metadatas = df.drop(columns=['full_text_description']).to_dict(orient='records')
# ChromaDB requires unique IDs for each document. Using product_id directly.
ids = df['product_id'].tolist()

# Generate embeddings in batches for efficiency (optional, but good practice)
embeddings = embedding_model.encode(documents, show_progress_bar=True).tolist()

# Add to ChromaDB
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print(f"Successfully indexed {collection.count()} products into ChromaDB.")
print("\n" + "-"*50 + "\n")

# --- 6. Demonstrate Semantic Search (Retrieval) --- 
print("6. Demonstrating semantic search...")
query_text = "I need something for my health and fitness tracking, maybe a gadget for my wrist."

# Embed the query
query_embedding = embedding_model.encode([query_text]).tolist()

# Query the ChromaDB collection
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2, # Retrieve top 2 most relevant products
    include=['documents', 'metadatas', 'distances']
)

print(f"Query: '{query_text}'\n")
print("Top retrieved products:")
for i, (doc, meta, dist) in enumerate(zip(results['documents'][0], results['metadatas'][0], results['distances'][0])):
    print(f"  Result {i+1} (Distance: {dist:.4f}):")
    print(f"    Product ID: {meta['product_id']}")
    print(f"    Name: {meta['name']}")
    print(f"    Category: {meta['category']}")
    print(f"    Description (indexed): {doc[:100]}...") # Show truncated indexed description
    print("\n")

print("Indexing and retrieval demonstration complete. The 'chroma_db' directory contains the persistent index.")


### Interpreting the Code Output and Performance Trade-offs

The code above demonstrates a complete, albeit simplified, pipeline for product catalog ingestion and vector indexing. Let's break down the output and discuss critical considerations:

**Interpreting the Output:**

1.  **Sample Data**: You'll first see the `DataFrame` containing our mock product data. This represents the raw, structured information we start with.
2.  **Prepared Text**: The `full_text_description` column shows how we concatenate relevant fields (`name`, `category`, `description`) into a single string. This combined text is what the embedding model will process to create a holistic semantic representation of each product.
3.  **Embedding Model Loading**: The `SentenceTransformer` model (`BAAI/bge-small-en-v1.5`) is loaded. This model converts text into numerical vectors.
4.  **ChromaDB Initialization**: We set up a local, persistent ChromaDB instance. The `get_or_create_collection` method ensures our `ecommerce_products` collection is ready.
5.  **Indexing Progress**: You'll see a progress bar as the embedding model processes each product's `full_text_description` and the resulting embeddings are added to ChromaDB. The final count confirms successful indexing.
6.  **Semantic Search Results**: This is the most crucial part. When we query with "I need something for my health and fitness tracking, maybe a gadget for my wrist," the system doesn't just look for keywords. Instead, it embeds the query and finds products whose embeddings are semantically closest to the query embedding. You'll observe that `Smartwatch Xtreme 2026` (P001) is likely the top result, even though the query didn't explicitly mention "smartwatch" or "Xtreme." The `distance` metric indicates how close the retrieved item's embedding is to the query embedding (lower distance means higher similarity).

**Performance Trade-offs and Considerations (2026 Context):**

1.  **Embedding Model Choice**: 
    *   **Accuracy vs. Speed/Cost**: Larger, more sophisticated models (e.g., `BAAI/bge-large-en-v1.5`, OpenAI's `text-embedding-3-large`, Google's `text-embedding-004`) offer superior semantic understanding but are slower and more expensive (if API-based). Smaller models (like `bge-small-en-v1.5` used here) are faster and cheaper, suitable for many applications where extreme nuance isn't required. 
    *   **Multimodal Embeddings**: For e-commerce, 2026 sees increasing adoption of multimodal embeddings that combine text, image, and even audio data. This allows for richer product representations and queries like "Show me a red dress similar to this image." This example focused on text for simplicity, but multimodal is a key future direction.

2.  **Vector Database Selection**: 
    *   **Managed vs. Self-Hosted**: Cloud-managed services (Pinecone, Weaviate Cloud, Qdrant Cloud, Azure AI Search, Vertex AI Vector Search) offer scalability, reliability, and ease of management but come with costs. Self-hosted options (Milvus, ChromaDB on your infrastructure, FAISS for in-memory) provide more control but require operational overhead. 
    *   **Scalability**: For millions of products, a distributed vector database is essential. ChromaDB, while excellent for local development, has scaling limitations compared to dedicated cloud solutions.
    *   **Filtering Capabilities**: Advanced vector databases allow filtering results based on metadata (e.g., "show me smartwatches under $300"). This is crucial for combining semantic search with structured queries.

3.  **Data Cleaning and Preprocessing**: 
    *   **Impact on Embeddings**: The quality of your raw text directly influences the quality of your embeddings. Poorly cleaned or noisy data will lead to less accurate embeddings and, consequently, poorer retrieval results. Invest heavily in robust ETL pipelines.
    *   **Text Augmentation**: Consider augmenting product descriptions with customer reviews, FAQs, or even user-generated content to provide richer context for embedding.

4.  **Indexing Strategy**: 
    *   **Batching**: Generating embeddings in batches (as shown in the code) is significantly more efficient than processing one document at a time.
    *   **Incremental Updates**: For dynamic catalogs, you'll need a strategy for updating or adding new products without re-indexing the entire database. Most vector databases support this.
    *   **Chunking Strategy**: For very long product descriptions or associated documents (e.g., user manuals), intelligent chunking (e.g., fixed size with overlap, semantic chunking) is vital to ensure relevant information isn't split across embeddings.

### Typical Use Cases for Agentic RAG in E-commerce:

*   **Intelligent Product Search**: Beyond keyword matching, allowing customers to describe what they want naturally.
*   **Personalized Recommendations**: Agents can retrieve products semantically similar to a user's past purchases or expressed preferences.
*   **Automated Customer Support**: Agents can quickly find answers to complex product-related questions from product manuals, FAQs, and reviews.
*   **Competitive Analysis**: Agents can analyze competitor product descriptions and features by embedding and comparing them.
*   **Dynamic Product Descriptions**: Agents can generate new, engaging product descriptions based on retrieved features and benefits.

By mastering data preparation, ingestion, and indexing, you lay the groundwork for a highly effective and intelligent e-commerce agent system.


### Resources

*   **ChromaDB Documentation**: [https://docs.trychroma.com/](https://docs.trychroma.com/)
*   **Sentence-Transformers Library**: [https://www.sbert.net/](https://www.sbert.net/)
*   **Hugging Face Models (BAAI/bge-small-en-v1.5)**: [https://huggingface.co/BAAI/bge-small-en-v1.5](https://huggingface.co/BAAI/bge-small-en-v1.5)
*   **Pandas Documentation**: [https://pandas.pydata.org/docs/](https://pandas.pydata.org/docs/)
*   **AutoGen Framework**: [https://microsoft.github.io/autogen/](https://microsoft.github.io/autogen/)
*   **Guide to RAG Data Preparation**: [https://www.pinecone.io/learn/rag-data-preparation/](https://www.pinecone.io/learn/rag-data-preparation/) (Pinecone blog, but concepts are general)
*   **Vector Databases Explained**: [https://www.databricks.com/glossary/vector-database](https://www.databricks.com/glossary/vector-database)
